## परिचय

इस पाठ में निम्न विषय कवर किए जाएंगे:
- फंक्शन कॉलिंग क्या है और इसके उपयोग के मामले
- Azure OpenAI का उपयोग करके फंक्शन कॉल कैसे बनाया जाता है
- फंक्शन कॉल को एक एप्लिकेशन में कैसे एकीकृत किया जाए

## शिक्षण लक्ष्य

इस पाठ को पूरा करने के बाद आप जानेंगे कि कैसे और समझेंगे:

- फंक्शन कॉलिंग का उपयोग करने का उद्देश्य
- Azure Open AI सेवा का उपयोग करके फंक्शन कॉल सेटअप करना
- अपने एप्लिकेशन के उपयोग मामले के लिए प्रभावी फंक्शन कॉल डिज़ाइन करना


## फ़ंक्शन कॉल की समझ  

इस पाठ के लिए, हम अपने शिक्षा स्टार्टअप के लिए एक फीचर बनाना चाहते हैं जो उपयोगकर्ताओं को तकनीकी पाठ्यक्रम खोजने के लिए चैटबॉट का उपयोग करने की अनुमति देता है। हम उनके कौशल स्तर, वर्तमान भूमिका और रुचि की तकनीक के अनुसार पाठ्यक्रमों की सिफारिश करेंगे।  

इसे पूरा करने के लिए हम निम्न का संयोजन उपयोग करेंगे:  
 - उपयोगकर्ता के लिए चैट अनुभव बनाने के लिए `Azure Open AI`  
 - उपयोगकर्ता की मांग के आधार पर पाठ्यक्रम खोजने में मदद करने के लिए `Microsoft Learn Catalog API`  
 - उपयोगकर्ता के प्रश्न को लेकर उसे API अनुरोध बनाने के लिए फ़ंक्शन को भेजने के लिए `Function Calling`  

शुरुआत करने के लिए, आइए देखें कि हम सबसे पहले फ़ंक्शन कॉलिंग का उपयोग क्यों करना चाहेंगे:  


### फंक्शन कॉलिंग क्यों

यदि आपने इस कोर्स में कोई अन्य पाठ पूरा किया है, तो आप शायद बड़े भाषा मॉडल (LLMs) का उपयोग करने की शक्ति समझते होंगे। उम्मीद है कि आप उनकी कुछ सीमाओं को भी देख सकते हैं।

फंक्शन कॉलिंग एक Azure Open AI सेवा की विशेषता है जो निम्नलिखित सीमाओं को दूर करने के लिए है:
1) निरंतर प्रतिक्रिया प्रारूप
2) चैट संदर्भ में आवेदन के अन्य स्रोतों से डेटा का उपयोग करने की क्षमता

फंक्शन कॉलिंग से पहले, LLM से प्रतिक्रियाएँ असंगठित और असंगत होती थीं। डेवलपर्स को जटिल सत्यापन कोड लिखना पड़ता था ताकि वे प्रतिक्रिया के प्रत्येक विभिन्न रूप को संभाल सकें।

उपयोगकर्ता "स्टॉकहोम में वर्तमान मौसम क्या है?" जैसे उत्तर प्राप्त नहीं कर सकते थे। ऐसा इसलिए था क्योंकि मॉडल उस समय तक प्रशिक्षित डेटा तक सीमित थे।

चलिए नीचे उदाहरण देखते हैं जो इस समस्या को दर्शाता है:

मान लीजिए हम छात्रों के डेटा का एक डेटाबेस बनाना चाहते हैं ताकि हम उन्हें सही पाठ्यक्रम सुझा सकें। नीचे हमारे पास दो छात्रों के विवरण हैं जो डेटा में बहुत समान हैं जो वे शामिल करते हैं।


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


हम इसे डेटा पार्स करने के लिए एक LLM को भेजना चाहते हैं। बाद में इसका उपयोग हमारे आवेदन में इसे किसी API को भेजने या डेटाबेस में संग्रहीत करने के लिए किया जा सकता है। 

चलिए दो समान प्रॉम्प्ट बनाते हैं जिनमें हम LLM को यह निर्देश देते हैं कि हम किस जानकारी में रुचि रखते हैं: 


हम इसे एक LLM को भेजना चाहते हैं ताकि वह हमारे उत्पाद के लिए महत्वपूर्ण भागों को पार्स कर सके। ताकि हम LLM को निर्देश देने के लिए दो समान प्रोम्प्ट बना सकें: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


इन दो प्रॉम्प्ट्स को बनाने के बाद, हम उन्हें `client.responses.create` का उपयोग करके LLM को भेजेंगे। हम प्रॉम्प्ट को `input` वेरिएबल में स्टोर करते हैं और भूमिका को `user` असाइन करते हैं। यह उपयोगकर्ता से संदेश चैटबोट को लिखे जाने की नकल करने के लिए है। 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

अब हम दोनों अनुरोधों को LLM को भेज सकते हैं और प्राप्त प्रतिक्रिया की जाँच कर सकते हैं। 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



हालांकि प्रॉम्प्ट एक जैसे हैं और विवरण भी समान हैं, फिर भी हमें `Grades` प्रॉपर्टी के विभिन्न प्रारूप मिल सकते हैं।

यदि आप ऊपर दिए गए सेल को कई बार चलाते हैं, तो प्रारूप `3.7` या `3.7 GPA` हो सकता है।

इसका कारण यह है कि LLM लिखित प्रॉम्प्ट के रूप में असंरचित डेटा लेता है और असंरचित डेटा ही लौटाता है। हमें एक संरचित प्रारूप चाहिए ताकि हमें पता हो कि इस डेटा को संग्रहित या उपयोग करते समय क्या अपेक्षा करनी है।

फंक्शन कॉलिंग का उपयोग करके, हम सुनिश्चित कर सकते हैं कि हमें संरचित डेटा वापस मिलें। फंक्शन कॉलिंग का उपयोग करते समय, LLM वास्तव में कोई फंक्शन नहीं चलाता या कॉल करता है। इसके बजाय, हम LLM के लिए एक संरचना बनाते हैं जिसे वह अपनी प्रतिक्रियाओं के लिए पालन करता है। फिर हम उन संरचित प्रतिक्रियाओं का उपयोग करके अपने अनुप्रयोगों में कौन सा फंक्शन चलाना है यह जानते हैं।
 


![फ़ंक्शन कॉलिंग फ़्लो डायग्राम](../../../../translated_images/hi/Function-Flow.083875364af4f4bb.webp)


हम तब फ़ंक्शन से जो वापस मिला है उसे ले सकते हैं और इसे LLM को वापस भेज सकते हैं। LLM तब उपयोगकर्ता के प्रश्न का उत्तर देने के लिए प्राकृतिक भाषा का उपयोग करके जवाब देगा। 


### फ़ंक्शन कॉल के उपयोग के मामले 

**बाहरी उपकरण कॉल करना** 
चैटबॉट उपयोगकर्ताओं से प्रश्नों के उत्तर प्रदान करने में उत्कृष्ट होते हैं। फ़ंक्शन कॉलिंग का उपयोग करके, चैटबॉट उपयोगकर्ताओं के संदेशों का उपयोग कुछ कार्यों को पूरा करने के लिए कर सकते हैं। उदाहरण के लिए, एक छात्र चैटबॉट से पूछ सकता है कि "मेरे शिक्षक को ईमेल भेजें कि मुझे इस विषय में अधिक सहायता चाहिए"। यह `send_email(to: string, body: string)` नामक फ़ंक्शन को कॉल कर सकता है।


**API या डेटाबेस क्वेरी बनाना**
उपयोगकर्ता प्राकृतिक भाषा का उपयोग करके जानकारी खोज सकते हैं जिसे एक स्वरूपित क्वेरी या API अनुरोध में बदला जाता है। इसका एक उदाहरण एक शिक्षक हो सकता है जो पूछता है "वे छात्र कौन हैं जिन्होंने अंतिम असाइनमेंट पूरा किया है" जो `get_completed(student_name: string, assignment: int, current_status: string)` नामक फ़ंक्शन को कॉल कर सकता है।


**संरचित डेटा बनाना**
उपयोगकर्ता एक टेक्स्ट ब्लॉक या CSV लेकर LLM का उपयोग करके उससे महत्वपूर्ण जानकारी निकाल सकते हैं। उदाहरण के लिए, एक छात्र शांति समझौतों के बारे में विकिपीडिया लेख को AI फ्लैश कार्ड बनाने के लिए परिवर्तित कर सकता है। यह `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नामक फ़ंक्शन का उपयोग करके किया जा सकता है।


## 2. आपका पहला फ़ंक्शन कॉल बनाना 

एक फ़ंक्शन कॉल बनाने की प्रक्रिया में 3 मुख्य चरण शामिल हैं: 
1. एक सूची के साथ Chat Completions API को कॉल करना जिसमें आपके फ़ंक्शन और एक उपयोगकर्ता संदेश हो 
2. एक्शन करने के लिए मॉडल की प्रतिक्रिया पढ़ना जैसे कि फ़ंक्शन या API कॉल को निष्पादित करना 
3. अपनी फ़ंक्शन से प्राप्त प्रतिक्रिया के साथ Chat Completions API को फिर से कॉल करना ताकि उस जानकारी का उपयोग करके उपयोगकर्ता को प्रतिक्रिया दी जा सके। 


![फंक्शन कॉल का प्रवाह](../../../../translated_images/hi/LLM-Flow.3285ed8caf4796d7.webp)


### एक फंक्शन कॉल के तत्व 

#### उपयोगकर्ता इनपुट 

पहला कदम एक उपयोगकर्ता संदेश बनाना है। इसे एक टेक्स्ट इनपुट का मान लेकर गतिशील रूप से दिया जा सकता है या आप यहाँ एक मान असाइन कर सकते हैं। यदि यह आपकी पहली बार चैट पूर्णता API के साथ काम कर रहे हैं, तो हमें संदेश की `role` और `content` को परिभाषित करना होगा। 

`role` या तो `system` (नियम बनाना), `assistant` (मॉडल) या `user` (अंत उपयोगकर्ता) हो सकता है। फंक्शन कॉलिंग के लिए, हम इसे `user` के रूप में असाइन करेंगे और एक उदाहरण प्रश्न देंगे। 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फ़ंक्शन बनाना।

अब हम एक फ़ंक्शन और उस फ़ंक्शन के पैरामीटर को परिभाषित करेंगे। हम यहाँ केवल एक फ़ंक्शन का उपयोग करेंगे जिसे `search_courses` कहा जाता है, लेकिन आप कई फ़ंक्शन बना सकते हैं।

**महत्वपूर्ण** : फ़ंक्शन LLM के सिस्टम संदेश में शामिल होते हैं और यह आपके उपलब्ध टोकन की मात्रा में शामिल किए जाएंगे।


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**परिभाषाएँ** 

`name` - वह फ़ंक्शन का नाम जिसे हम कॉल करना चाहते हैं। 

`description` - यह फ़ंक्शन कैसे काम करता है इसका विवरण। यहाँ विशिष्ट और स्पष्ट होना महत्वपूर्ण है 

`parameters` - उन मानों और प्रारूपों की सूची जो आप चाहते हैं कि मॉडल अपनी प्रतिक्रिया में उत्पन्न करे 


`type` - उन गुणों का डेटा प्रकार जिनमें भंडारण किया जाएगा 

`properties` - विशिष्ट मानों की सूची जिन्हें मॉडल अपनी प्रतिक्रिया के लिए उपयोग करेगा 


`name` - वह गुण का नाम जिसे मॉडल अपनी स्वरूपित प्रतिक्रिया में उपयोग करेगा 

`type` - इस गुण का डेटा प्रकार 

`description` - विशिष्ट गुण का विवरण 


**वैकल्पिक**

`required` - फ़ंक्शन कॉल को पूरा करने के लिए आवश्यक गुण 


### फ़ंक्शन कॉल करना  
एक फ़ंक्शन परिभाषित करने के बाद, अब हमें इसे Chat Completion API को कॉल में शामिल करना होगा। हम यह `functions` को रिक्वेस्ट में जोड़कर करते हैं। इस स्थिति में `functions=functions` होता है।  

एक विकल्प यह भी है कि `function_call` को `auto` पर सेट किया जाए। इसका मतलब है कि हम खुद से फ़ंक्शन असाइन करने के बजाय उपयोगकर्ता संदेश के आधार पर LLM को तय करने देंगे कि कौन सा फ़ंक्शन कॉल किया जाना चाहिए।  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


अब चलिए प्रतिक्रिया को देखते हैं और देखते हैं कि इसे कैसे स्वरूपित किया गया है: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

आप देख सकते हैं कि फ़ंक्शन का नाम बुलाया गया है और उपयोगकर्ता संदेश से, LLM उस डेटा को खोजने में सक्षम था जो फ़ंक्शन के तर्कों से मेल खाता है। 


## 3.एक एप्लिकेशन में फ़ंक्शन कॉल्स को एकीकृत करना। 


LLM से प्राप्त फॉर्मेटेड प्रतिक्रिया का परीक्षण करने के बाद, अब हम इसे एक एप्लिकेशन में एकीकृत कर सकते हैं। 

### प्रवाह का प्रबंधन 

इसे हमारे एप्लिकेशन में एकीकृत करने के लिए, आइए निम्नलिखित कदम उठाएँ: 

पहले, Open AI सेवाओं को कॉल करें और संदेश को `response_message` नामक एक वेरिएबल में सहेजें। 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


अब हम वह फ़ंक्शन परिभाषित करेंगे जो Microsoft Learn API को कॉल करेगा ताकि पाठ्यक्रमों की सूची प्राप्त की जा सके: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वोत्तम अभ्यास के रूप में, हम फिर देखेंगे कि क्या मॉडल किसी फंक्शन को कॉल करना चाहता है। इसके बाद, हम उपलब्ध फंक्शनों में से एक बनाएंगे और उसे उस फंक्शन से मेल करेंगे जिसे कॉल किया जा रहा है।
फिर हम फंक्शन के आर्ग्यूमेंट्स लेंगे और उन्हें LLM के आर्ग्यूमेंट्स से मैप करेंगे।

अंत में, हम फंक्शन कॉल संदेश और `search_courses` संदेश द्वारा लौटाए गए मान जोड़ देंगे। इससे LLM को वह सारी जानकारी मिल जाती है जिसकी उसे आवश्यकता होती है
उपयोगकर्ता को प्राकृतिक भाषा में जवाब देने के लिए।


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


अब हम अपडेट किया गया संदेश LLM को भेजेंगे ताकि हमें API JSON प्रारूपित प्रतिक्रिया के बजाय प्राकृतिक भाषा प्रतिक्रिया मिल सके।


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड चुनौती 

शानदार काम! Azure Open AI फ़ंक्शन कॉलिंग की अपनी सीख जारी रखने के लिए आप निर्माण कर सकते हैं: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फ़ंक्शन के और भी पैरामीटर जो शिक्षार्थियों को और कोर्स खोजने में मदद कर सकते हैं। आप उपलब्ध API पैरामीटर यहाँ पा सकते हैं: 
 - एक और फ़ंक्शन कॉल बनाएं जो शिक्षार्थी से उनकी मातृभाषा जैसी अधिक जानकारी ले 
 - त्रुटि हैंडलिंग बनाएं जब फ़ंक्शन कॉल और/या API कॉल कोई उपयुक्त कोर्स वापस न करे 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
